In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('data/original_data.csv')

In [ ]:
numeric_cols = [
    'release_speed',
    'release_pos_x',
    'release_pos_y',
    'release_pos_z',
    'pfx_x',
    'pfx_z',
    'vx0',
    'vy0',
    'vz0',
    'ax',
    'ay',
    'az',
    'release_spin_rate',
    'spin_axis',
    'arm_angle'
]
categorical_cols = [
    'p_throws',
    'pitch_type'
]

In [ ]:
encoder = LabelEncoder()
df['pitch_type'] = encoder.fit_transform(df['pitch_type'])
mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
print(mapping)

In [ ]:
df = df[['game_date'] + numeric_cols + categorical_cols]
df.dropna(inplace=True)
df['p_throws'] = np.where(df['p_throws'] == 'R', 1, 0)

df['game_date'] = pd.to_datetime(df['game_date'])

In [ ]:
scaler = StandardScaler()
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), numeric_cols),
        ('categorical', 'passthrough', categorical_cols)
    ],
    verbose_feature_names_out=False
).set_output(transform='pandas')


In [ ]:
train_val_dfs = df[df['game_date'] <= '2026-07-20']
train_val_dfs.drop(columns='game_date', inplace=True)

train_df, validation_df = train_test_split(
    train_val_dfs, test_size=0.05, random_state=42
)

In [ ]:
train_df = preprocessor.fit_transform(train_df)
validation_df = preprocessor.transform(validation_df)

In [ ]:
test_df = df[df['game_date'] > '2026-07-20']
test_df.drop(columns='game_date', inplace=True)
test_df = preprocessor.transform(test_df)

In [ ]:
train_df.to_csv('data/training.csv', index=False)
validation_df.to_csv('data/validation.csv', index=False)
test_df.to_csv('data/testing.csv', index=False)